In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
capture_path = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/'
index_df = pd.read_csv(capture_path + 'index.csv')
print(index_df.shape)

nrows = index_df.shape[0]

# index_df expand by creating duplicate rows for each row by adding a new column 'cam': ['cam0','cam1','cam3']
index_df = index_df.loc[index_df.index.repeat(3)].reset_index(drop=True)
index_df['cam'] = ['cam0','cam1','cam2']*nrows

index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(155, 4)
(465, 5)


,video_id,template,placeholders,session,cam
0,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0
1,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1


In [5]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(465, 5)


,name,label,top1_pred,correct_top1,correct_top5
312,260930-143609_cam0.avi,112,112,True,True
144,260922-151624_cam0.avi,112,114,False,False


In [6]:
# I want to break video_id into video_id and cam
result[['video_id', 'cam']] = result.name.str.split('.').str[0].str.split('_', expand=True)
print(result.shape)

del result['name']

result.head(2)


(465, 7)


,label,top1_pred,correct_top1,correct_top5,video_id,cam
0,49,49,True,True,260922-134214,cam0
1,49,49,True,True,260922-134214,cam1


In [7]:
# Bring in columns with a left join
result = pd.merge(result, index_df, how = 'left', on = ['video_id','cam']) # LEFT JOIN
print(result.shape)
result.sample(3)

(465, 9)


,label,top1_pred,correct_top1,correct_top5,video_id,cam,template,placeholders,session
75,107,107,True,True,260922-144349,cam0,Putting [something] next to [something],"phone, water bottle",books_2
269,112,106,False,True,260930-142029,cam2,Putting [something] onto [something],"magnet, plate",magnet_2
447,37,94,False,True,261008-114837,cam0,Moving [something] and [something] closer to each other,"nothing, nothing",mime_moveCloserToEachOther


In [8]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (465, 174)
d

(465, 174)


NpzFile './predictions.npz' with keys: logits, probabilities

In [9]:
probs = d['probabilities']
d = {'probs': probs}
d["probs"].shape

(465, 174)

# main result

In [10]:

n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()

print(f"Number of samples: {n}")
print(f"Top-1 accuracy: {top1_acc:.4f}")
print(f"Top-5 accuracy: {top5_acc:.4f}")

Number of samples: 465
Top-1 accuracy: 0.5527
Top-5 accuracy: 0.8602


- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [11]:
result.groupby(['cam']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
cam,,,
cam0,155,0.53,0.83
cam1,155,0.66,0.95
cam2,155,0.46,0.79


In [12]:
# result[result.cam=='cam1'].groupby(['template']).agg(
#     count=('video_id', 'count'),
#     top1_acc=('correct_top1', 'mean'),
#     top5_acc=('correct_top5', 'mean')
# )

In [13]:
result.session.unique()

array(['plug_1', 'plug_2', 'magnet_1', 'magnet_2', 'books_1', 'books_2',
       'lady_1', 'lady_2', 'paper_1', 'paper_2', 'plugging_1',
       'showing_photo_of_1', 'attaching_1', 'piling_1',
       'mime_pretend_poke', 'pretend_poke_1', 'mime_pretend_sprinkle',
       'pretend_sprinkle_1', 'mime_pretend_pickUp', 'pretend_pickUp_1',
       'mime_bending', 'bending_1', 'mime_moveCloserToEachOther',
       'moveCloserToEachOther_1'], dtype=object)

In [14]:
# temp = result[result.cam=='cam1']

# temp

# temp.groupby(np.where(temp['session'].str.startswith('mime'), 'mime', 'control') ).agg(
#     count=('video_id', 'count'),
#     top1_acc=('correct_top1', 'mean'),
#     top5_acc=('correct_top5', 'mean')
# )

# Main Result by Specific Session

\#\#\#\# session 1

* **plug\_1** – 5 vids special  action:’Plugging something with something’ with plug object  
* **plug\_2** – 5 vids common actions with other objects – Plug is **distractor** lying around on table.  
* **magnet\_1**  
* **magnet\_2**  
* **books\_1**  
* **books\_2**  
* **lady\_1**  
* **lady\_2**  
* **paper\_1**  
* **paper\_2**

\#\#\#\# session 2

* **Plug\_2** – 15 vids of common actions using plug as object-plug is **distractor** trying to spike prob of special action.  
* **Magnet\_2** – 15 vids …  
* **Books\_2** – …  
* **lady\_2**  
*   
* **plugging\_1** –  3 vids of special action:plugging with objects other than plug  
* **showing\_photo\_of\_1** – 3 vids of special action:showing\_photo\_of\_something  with objects other than photo\_of\_lady  
* **Attaching\_1** – …  
* **piling\_1**  
  


\#\#\#\# session 3

* **mime\_pretend\_poke** – 3 vids of action:pretending\_to\_poke\_something with nothing  
* **pretend\_poke\_1** – 3 vids of action:pretending\_to\_poke\_something with actual objects  
*   
* **mime\_pretend\_sprinkle** – …  
* **pretend\_sprinkle\_1** –   
*   
* **mime\_pretend\_pickUp** –   
* **pretend\_pickUp\_1**  
*   
* **mime\_bending** – 3 vids of action:’Bending \[something\] so that it deforms’ with nothing  
* **bending\_1** – 3 vids of action:’Bending \[something\] so that it deforms’ with actual objects  
*   
* **mime\_moveCloserToEachOther …**  
* **moveCloserToEachOther\_1**



In [15]:
result['session_special'] = np.where(result.session.isin(['plug_2','magnet_2','lady_2','books_2']), 'distractors_in_video',
                               np.where(result.session.isin(['plug_1','magnet_1','lady_1','books_1']), 'session_1_control', 
                                        np.where(result.session.isin(['plugging_1','showing_photo_of_1','attaching_1','piling_1']), 'session2_specialActions_commonobjects',
                                                 np.where(result['session'].str.startswith('mime'), 'miming_videos', 

                                                          'mime_actions_control_videos'))))

result.tail(2)

,label,top1_pred,correct_top1,correct_top5,video_id,cam,template,placeholders,session,session_special
463,37,37,True,True,261008-114959,cam1,Moving [something] and [something] closer to each other,"pen, board marker",moveCloserToEachOther_1,mime_actions_control_videos
464,37,36,False,True,261008-114959,cam2,Moving [something] and [something] closer to each other,"pen, board marker",moveCloserToEachOther_1,mime_actions_control_videos


In [16]:
temp = result[result.cam=='cam1']

temp.groupby(['session_special']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean'),
    unique_sessions=('session', lambda x: ','.join(map(str, x.unique()))),
)

,count,top1_acc,top5_acc,unique_sessions
session_special,,,,
distractors_in_video,80,0.72,0.99,"plug_2,magnet_2,books_2,lady_2"
mime_actions_control_videos,25,0.72,1.00,"paper_1,paper_2,pretend_poke_1,pretend_sprinkle_1,pretend_pickUp_1,bending_1,moveCloserToEachOther_1"
miming_videos,18,0.39,0.72,"mime_pretend_poke,mime_pretend_sprinkle,mime_pretend_pickUp,mime_bending,mime_moveCloserToEachOther"
session2_specialActions_commonobjects,12,0.67,1.00,"plugging_1,showing_photo_of_1,attaching_1,piling_1"
session_1_control,20,0.60,0.95,"plug_1,magnet_1,books_1,lady_1"


In [20]:
import warnings
warnings.filterwarnings('ignore')

temp['date'] = pd.to_datetime(temp['video_id'].str.split('-').str[0], format='%y%m%d')
temp['date'] = temp['date'].dt.date

temp.groupby(['date']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean'),
    unique_sessions=('session', lambda x: ','.join(map(str, x.unique()))),
)

,count,top1_acc,top5_acc,unique_sessions
date,,,,
2026-09-22,50,0.62,0.96,"plug_1,plug_2,magnet_1,magnet_2,books_1,books_2,lady_1,lady_2,paper_1,paper_2"
2026-09-30,72,0.74,1.00,"plug_2,lady_2,magnet_2,books_2,plugging_1,showing_photo_of_1,attaching_1,piling_1"
2026-10-08,33,0.58,0.85,"mime_pretend_poke,pretend_poke_1,mime_pretend_sprinkle,pretend_sprinkle_1,mime_pretend_pickUp,pretend_pickUp_1,mime_bending,bending_1,mime_moveCloserToEachOther,moveCloserToEachOther_1"


# Granular analysis

##### id2label

In [ ]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}

print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [18]:
# Apply mapping to create new column
result['template_custom'] = result['label'].astype(str).map(id2label_dict)
result.sample(3)

,label,top1_pred,correct_top1,correct_top5,video_id,cam,template,placeholders,session,session_special,template_custom
246,107,107,True,True,260930-141829,cam0,Putting [something] next to [something],"pen, magnet",magnet_2,distractors_in_video,Putting [something] next to [something]
21,45,27,False,True,260922-141308,cam0,Moving [something] up,wallet,plug_2,distractors_in_video,Moving [something] up
76,107,107,True,True,260922-144349,cam1,Putting [something] next to [something],"phone, water bottle",books_2,distractors_in_video,Putting [something] next to [something]


In [19]:
pd.testing.assert_series_equal(result.template, result.template_custom, check_names=False, check_exact=False)

In [20]:
np.mean(result.template == result.template_custom)*100

np.float64(100.0)

In [21]:
df = result[['video_id',	'cam'	,'session'	,'placeholders','template',	'label'	,'top1_pred','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(465, 9)


,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True
1,260922-134214,cam1,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True


In [29]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5,prob_0,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000008,0.000000,0.000000,0.000024
1,260922-134214,cam1,plug_1,"plug, electrical extension",Plugging [something] into [something],49,49,True,True,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000266,0.000000,0.000000,0.000001


In [ ]:
pd.set_option('display.float_format',  '{:,.4f}'.format)

In [34]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [35]:
print(df.template.unique())

['Plugging [something] into [something]'
 'Putting [something] next to [something]'
 'Putting [something] on a surface' 'Moving [something] up'
 'Putting [something] onto [something]'
 'Pushing [something] with [something]'
 'Attaching [something] to [something]' 'Piling [something] up'
 'Showing a photo of [something] to the camera'
 'Tearing [something] into two pieces' 'Pretending to poke [something]'
 'Pretending to sprinkle air onto [something]'
 'Pretending to pick [something] up'
 'Bending [something] so that it deforms'
 'Moving [something] and [something] closer to each other']


In [36]:
avg_prob_in_diff_cases(df, session_type='plug_2', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug_2 =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc  top5acc                                     examples
0  plug_2      20   0.041898  0.185516 0.600000 0.900000  260922-141236, 260922-141249, 260922-141308
==================== Videos ====================
         video_id   cam session                placeholders                                 template label top1_pred correct_top1 correct_top5 prob_107 prob_114 prob_108 prob_133 prob_103
16  260922-141236  cam1  plug_2  water bottle, board marker  Putting [something] next to [something]   107       107         True         True 0.886281 0.046515 0.040475 0.015378 0.003141

Top 5 Probability Breakdown:
  prob_107 ->  0.88628  |  Putting [something] next to [something]
  prob_114 ->  0.04652  |  Putting [something similar to other things that are already on the table]
  prob_108 ->  0.04047  |  Putting [something]